<a href="https://colab.research.google.com/github/Kumkum15/rating-prediction-via-prompting/blob/main/Rating_prediction_via_prompting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install required libraries

In [13]:
!pip install openai pandas tqdm

import pandas as pd
import json
from tqdm import tqdm
from openai import OpenAI


Set Up OpenRouter API Client

In [42]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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"
)

Load Dataset & Sample 200 Rows

In [43]:
df = pd.read_csv("/content/yelp.csv")

# Keep only review text + true stars
df = df[["text", "stars"]]

# Sample ~200 rows for testing
sample_df = df.sample(200, random_state=42).reset_index(drop=True)

sample_df.head()


,text,stars
0,We got here around midnight last Friday... the...,4
1,Brought a friend from Louisiana here. She say...,5
2,"Every friday, my dad and I eat here. We order ...",3
3,"My husband and I were really, really disappoin...",1
4,Love this place! Was in phoenix 3 weeks for w...,5


Define 3 Prompt Versions

In [70]:
prompt_v1 = """
You are a strict JSON generator.

Classify this Yelp review into a star rating from 1 to 5.

Rules:
- Output ONLY JSON (no markdown, no backticks).
- Follow this schema exactly:

{{"predicted_stars": 0, "explanation": ""}}

Replace with real values.

Review:
"{review}"
"""


In [71]:
prompt_v2 = """
You must output ONLY valid JSON. No Markdown, no backticks.

Star meanings:
1 = very negative
2 = negative
3 = neutral
4 = positive
5 = very positive

Review:
"{review}"

Return exactly this JSON format:

{{"predicted_stars": <integer>, "explanation": "<short reason>"}}
"""


In [72]:
prompt_v3 = """
You are an expert review analyst.

INTERNAL steps (do not reveal):
1. Analyze sentiment
2. Map sentiment → rating 1–5
3. Summarize brief reasoning

OUTPUT RULES:
- Output ONLY JSON
- No markdown or backticks

REVIEW:
"{review}"

FINAL OUTPUT:
{{"predicted_stars": <integer>, "explanation": "<brief reason>"}}
"""


Create LLM Call Function

In [79]:
def call_llm(review, prompt):
    full_prompt = prompt.format(review=review)

    response = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[{"role": "user", "content": full_prompt}],
        temperature=0,
        extra_headers={
            "HTTP-Referer": "https://colab.research.google.com",
            "X-Title": "Yelp Prompting Notebook"
        }
    )

    raw_text = response.choices[0].message.content.strip()

    try:
        return json.loads(raw_text), True
    except:
        return {"predicted_stars": None, "explanation": raw_text}, False


In [80]:
resp = client.chat.completions.create(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": "Say hi in JSON"}],
    extra_headers={
        "HTTP-Referer": "https://colab.research.google.com",
        "X-Title": "Testing"
    }
)

print(resp.choices[0].message.content)


Sure! Here’s a simple JSON representation of a greeting:

```json
{
  "greeting": "Hi"
}
```


In [81]:
test_review = "The food was amazing but service was slow."
resp, valid = call_llm(test_review, prompt_v1)
resp, valid

({'predicted_stars': 4,
  'explanation': 'The review highlights that the food was amazing, which is a strong positive aspect, but it also mentions slow service, which is a negative aspect. Overall, the positive experience with the food outweighs the negative experience with the service, leading to a 4-star rating.'},
 True)

Evaluation Loop Function

In [82]:
def evaluate_prompt(prompt, data):
    results = []

    for i, row in tqdm(data.iterrows(), total=len(data)):
        review = row["text"]
        true_rating = row["stars"]

        response, valid_json = call_llm(review, prompt)

        predicted = response.get("predicted_stars", None)

        results.append({
            "review": review,
            "true_rating": true_rating,
            "predicted": predicted,
            "valid_json": valid_json,
            "explanation": response.get("explanation", "")
        })

    return pd.DataFrame(results)


Run Evaluation for Prompt 1

In [83]:
results_v1 = evaluate_prompt(prompt_v1, sample_df)
results_v1.head()

100%|██████████| 200/200 [04:51<00:00,  1.46s/it]


,review,true_rating,predicted,valid_json,explanation
0,We got here around midnight last Friday... the...,4,4,True,The review highlights positive aspects such as...
1,Brought a friend from Louisiana here. She say...,5,5,True,The review expresses high satisfaction with th...
2,"Every friday, my dad and I eat here. We order ...",3,4,True,The review expresses a positive experience wit...
3,"My husband and I were really, really disappoin...",1,1,True,The review expresses significant disappointmen...
4,Love this place! Was in phoenix 3 weeks for w...,5,5,True,The reviewer expresses strong positive feeling...


Run Evaluation for Prompt 2

In [84]:
results_v2 = evaluate_prompt(prompt_v2, sample_df)
results_v2.head()

100%|██████████| 200/200 [03:47<00:00,  1.14s/it]


,review,true_rating,predicted,valid_json,explanation
0,We got here around midnight last Friday... the...,4,4,True,The review highlights positive aspects such as...
1,Brought a friend from Louisiana here. She say...,5,5,True,The review expresses a very positive sentiment...
2,"Every friday, my dad and I eat here. We order ...",3,4,True,The review expresses a positive experience wit...
3,"My husband and I were really, really disappoin...",1,1,True,The review expresses extreme disappointment wi...
4,Love this place! Was in phoenix 3 weeks for w...,5,5,True,The review expresses strong enthusiasm and sat...


Run Evaluation for Prompt 3

In [85]:
results_v3 = evaluate_prompt(prompt_v3, sample_df)
results_v3.head()

100%|██████████| 200/200 [04:51<00:00,  1.46s/it]


,review,true_rating,predicted,valid_json,explanation
0,We got here around midnight last Friday... the...,4,4,True,The review highlights positive aspects such as...
1,Brought a friend from Louisiana here. She say...,5,5,True,The review expresses high satisfaction with th...
2,"Every friday, my dad and I eat here. We order ...",3,4,True,The review expresses positive sentiments about...
3,"My husband and I were really, really disappoin...",1,1,True,The reviewer expresses significant disappointm...
4,Love this place! Was in phoenix 3 weeks for w...,5,5,True,The review expresses strong positive sentiment...


Compute Metrics Function

In [86]:
def compute_metrics(df):
    # Accuracy (ignore cases where predicted is None)
    df_valid_pred = df[df["predicted"].notnull()]

    accuracy = (df_valid_pred["predicted"] == df_valid_pred["true_rating"]).mean()

    json_rate = df["valid_json"].mean()

    return accuracy, json_rate

Calculate Metrics for All 3 Prompts

In [87]:
acc1, json1 = compute_metrics(results_v1)
acc2, json2 = compute_metrics(results_v2)
acc3, json3 = compute_metrics(results_v3)

comparison = pd.DataFrame({
    "Prompt Version": ["Prompt 1", "Prompt 2", "Prompt 3"],
    "Accuracy": [acc1, acc2, acc3],
    "JSON Validity": [json1, json2, json3]
})

comparison

,Prompt Version,Accuracy,JSON Validity
0,Prompt 1,0.685,1.0
1,Prompt 2,0.655,1.0
2,Prompt 3,0.645,1.0


Save Results to CSV

In [88]:
results_v1.to_csv("results_prompt1.csv", index=False)
results_v2.to_csv("results_prompt2.csv", index=False)
results_v3.to_csv("results_prompt3.csv", index=False)

comparison.to_csv("prompt_comparison.csv", index=False)

Final Observations

After evaluating all three prompt strategies on a sample of 200 Yelp reviews using the OpenRouter LLM, we compared performance across Accuracy and JSON Validity Rate. The results are:

Prompt 1: Accuracy - 0.685	JSON Validity - 1.0

Prompt 2:	Accuracy - 0.655	JSON Validity - 1.0

Prompt 3:	Accuracy - 0.645	JSON Validity - 1.0

1. JSON Validity (All Prompts = 100%)

A major goal in this task is ensuring structured JSON output.
All three prompts achieved a 100% JSON validity rate, meaning:

The prompt instructions were strong enough

The model consistently followed strict JSON schemas

No malformed outputs or markdown issues occurred

This confirms that the addition of rules like “Output ONLY JSON”, no backticks, and strict schemas worked successfully.

2. Accuracy Comparison

Accuracy varied slightly across the prompts:

Prompt 1 achieved the highest accuracy (68.5%)

Prompt 2 followed with 65.5%

Prompt 3 performed slightly lower at 64.5%

Even though Prompt 3 is usually expected to perform best because of step-by-step internal reasoning. But in the dataset Prompt 1 performed better.

This minor accuracy variation can happen due to:

Subtle prompt wording differences

Model bias toward certain sentiment interpretations

How the LLM handles borderline 3-star vs 4-star reviews

3. Why Prompt 1 Outperformed the Others

Prompt 1, although the “simpler” prompt, performed best likely because:

It gave the model more freedom to interpret sentiment

It didn’t over-constrain the reasoning process

Yelp reviews often contain mixed sentiment (positive food + negative service), and a less rigid prompt sometimes handles this better

This aligns with known LLM behavior:

Too much structure sometimes suppresses nuance in sentiment scoring.

4. Prompt 2 Performance

Prompt 2 adds structured guidelines (1–5 mapping rules).
This increases consistency but can reduce flexibility → leading to slightly lower accuracy than Prompt 1.

5. Prompt 3 Performance

Prompt 3 forces more deliberate internal reasoning, which generally improves consistency.
However, in this case, the stricter structure slightly reduced accuracy. This happens when:

The prompt encourages conservative or midpoint ratings

Over-thinking reduces alignment with human-assigned Yelp star ratings

Still, it remained highly reliable and fully valid in JSON output.

Final Conclusion

Prompt 1 is the best choice for this dataset because it produced the highest accuracy while maintaining perfect JSON structure.

All three prompts are reliable, they produced 100% valid JSON, which is crucial for downstream evaluation.

Prompt variation affects sentiment interpretation even with the same model and dataset.

Prompt design influences model behavior, showing that carefully balancing structure and flexibility leads to optimal results.

Recommended Prompt for Production

If the goal is accuracy, choose Prompt 1.
If the goal is maximum reliability + consistency, choose Prompt 3.